In [29]:
import os
from dotenv import load_dotenv
load_dotenv()
import json 


import numpy as np 
import pandas as pd 
from pathlib import Path
from datasets import load_dataset
import subprocess

from huggingface_hub import hf_hub_download

In [27]:
# Project root
PROJECT_ROOT = Path.cwd().parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

# Create directory if it doesn't exist
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

In [24]:
print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data directory: {RAW_DATA_DIR}")

Project root: d:\ai_project\FinGuide-AI
Raw data directory: d:\ai_project\FinGuide-AI\data\raw


In [18]:
from huggingface_hub import list_repo_files

files = list_repo_files(
    repo_id="ibm-research/finqa",
    repo_type="dataset"
)

print("\n".join(files))

.gitattributes
README.md
finqa.py


In [ ]:
finqa_path = Path("../data/raw/FinQA")

for file in finqa_path.rglob("*"):
    if file.is_file():
        print(file)

In [37]:
finqa_dataset_path = Path("../data/raw/FinQA/dataset")

with open(finqa_dataset_path/'train.json' , 'r' , encoding='utf-8') as f :
    finqa_train = json.load(f)
with open(finqa_dataset_path/'dev.json' , 'r' , encoding='utf-8') as f :
    finqa_dev = json.load(f)
with open(finqa_dataset_path/'test.json' , 'r' , encoding='utf-8') as f :
    finqa_test = json.load(f)

In [39]:
print('Train: ' , len(finqa_train))
print('dev: ' , len(finqa_dev))
print('test: ' , len(finqa_test))

Train:  6251
dev:  883
test:  1147


In [41]:
examples = finqa_train[0]
print(examples.keys())

dict_keys(['pre_text', 'post_text', 'filename', 'table_ori', 'table', 'qa', 'id', 'table_retrieved', 'text_retrieved', 'table_retrieved_all', 'text_retrieved_all'])


In [46]:
examples['filename']

'ADI/2009/page_49.pdf'

In [48]:
from pprint import pprint

pprint(examples)

{'filename': 'ADI/2009/page_49.pdf',
 'id': 'ADI/2009/page_49.pdf-1',
 'post_text': ['fair value of forward exchange contracts after a 10% ( 10 % ) '
               'unfavorable movement in foreign currency exchange rates asset '
               '( liability ) .',
               '.',
               '.',
               '.',
               '.',
               '.',
               '.',
               '.',
               '.',
               '$ 20132 $ ( 9457 ) fair value of forward exchange contracts '
               'after a 10% ( 10 % ) favorable movement in foreign currency '
               'exchange rates liability .',
               '.',
               '.',
               '.',
               '.',
               '.',
               '.',
               '.',
               '.',
               '.',
               '.',
               '.',
               '.',
               '.',
               '.',
               '.',
               '.',
               '.',
               '.',
               

In [50]:
qa = examples['qa']

In [53]:
qa

{'question': 'what is the the interest expense in 2009?',
 'answer': '380',
 'explanation': '',
 'ann_table_rows': [],
 'ann_text_rows': [1],
 'steps': [{'op': 'divide1-1', 'arg1': '100', 'arg2': '100', 'res': '1%'},
  {'op': 'divide1-2', 'arg1': '3.8', 'arg2': '#0', 'res': '380'}],
 'program': 'divide(100, 100), divide(3.8, #0)',
 'gold_inds': {'text_1': 'if libor changes by 100 basis points , our annual interest expense would change by $ 3.8 million .'},
 'exe_ans': 3.8,
 'tfidftopn': {'text_14': 'dollar , would have on the fair value of our forward exchange contracts as of october 31 , 2009 and november 1 , 2008: .',
  'text_0': 'interest rate to a variable interest rate based on the three-month libor plus 2.05% ( 2.05 % ) ( 2.34% ( 2.34 % ) as of october 31 , 2009 ) .'},
 'program_re': 'divide(3.8, divide(100, 100))',
 'model_input': [['text_0',
   'interest rate to a variable interest rate based on the three-month libor plus 2.05% ( 2.05 % ) ( 2.34% ( 2.34 % ) as of october 31 , 2

In [54]:
import random

for i in random.sample(range(len(finqa_train)), 3):
    qa = finqa_train[i]["qa"]

    print("=" * 80)
    print("Question:", qa["question"])
    print("Answer:", qa["answer"])
    print("Executable Answer:", qa.get("exe_ans"))
    print("Program:", qa.get("program"))

Question: as of december 31 , 2017 what was the percent of the utilization of the allowed letter of credit limit on the entergy texas
Answer: 85.3%
Executable Answer: 0.85333
Program: divide(25.6, 30)
Question: what portion of the change in net income during 2016 was related the irs audit?
Answer: 77.6%
Executable Answer: 0.77594
Program: divide(136.1, 175.4)
Question: what is the net of increases related to tax positions taken during a prior period and decreases related to tax positions taken during a prior period , in millions?
Answer: -43
Executable Answer: -43.0
Program: add(27, -70)


In [55]:
import re
from collections import Counter

def classify_answer(answer):
    answer = str(answer).strip()

    if "%" in answer:
        return "percentage"

    if re.fullmatch(r"-?\d+", answer.replace(",", "")):
        return "integer"

    if re.fullmatch(r"-?\d+\.\d+", answer.replace(",", "")):
        return "decimal"

    return "other"


answer_types = Counter(
    classify_answer(item["qa"]["answer"])
    for item in finqa_train
)

answer_types

Counter({'percentage': 3508, 'integer': 1248, 'decimal': 1192, 'other': 303})

In [56]:
for item in finqa_train:
    answer = item["qa"]["answer"]

    if classify_answer(answer) == "other":
        print("Question:", item["qa"]["question"])
        print("Answer:", answer)
        print("Program:", item["qa"].get("program"))
        print("-" * 80)

Question: during the 2012 year , did the equity awards in which the prescribed performance milestones were achieved exceed the equity award compensation expense for equity granted during the year?
Answer: 
Program: multiply(607, 18.13), multiply(#0, const_1000), multiply(3.3, const_1000000), greater(#1, #2)
--------------------------------------------------------------------------------
Question: in millions , what is the total of home equity lines of credit?
Answer: 
Program: add(15553, 7376)
--------------------------------------------------------------------------------
Question: what was the total amount lost from the bond authorization to the withdrawn?
Answer: $ 13 million
Program: subtract(78.0, 75.3), subtract(58.0, 49.9), subtract(54.0, 51.8), add(#0, #1), add(#2, #3)
--------------------------------------------------------------------------------
Question: did jpmorgan chase outperform the kbw bank index?
Answer: yes
Program: greater(189.57, 137.82)
--------------------------